# Week 13 — Multi-Agent RL and AlphaZero · Workbook

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


**A bounded first investigation:** Label every search value with its player. Start with **Exercises 1, 2 and 4** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-label-every-search-value-with-its-player).
Save your own prediction before running the reference cells. If you implement an exercise,
replace its reference call and verify that its check invokes your function.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
import torch
from project import solutions as sol


## Exercise 1 — PUCT formula (★)

Implement the PUCT score: $-Q(c) + c_{\text{puct}} \cdot P(c) \cdot \sqrt{N(\text{parent})} / (1 + N(c))$.

> **Hint**: Q is negated because the child's value is from the opponent's POV.

In [ ]:
s = sol.exercise_1_puct(prior=0.5, parent_visits=100, child_visits=10, q=0.2, c_puct=1.5)
print('PUCT =', round(s, 3))
assert s > 0


## Exercise 2 — Terminal value (★)

Given a Tic-Tac-Toe board (canonical form: +1 = player to move, -1 = opponent),
return the terminal value from the current player's POV or `None` if the
game is ongoing.

> **Hint**: see `project.tictactoe.TicTacToe.terminal_value`.

In [ ]:
import numpy as np
state = np.zeros((3, 3), dtype=np.int8)
print('empty board:', sol.exercise_2_terminal_value(state))
state[0, :] = -1
print('row of O (just-moved player won):', sol.exercise_2_terminal_value(state))


## Exercise 3 — D4 symmetries (★)

Tic-Tac-Toe has 4 rotations × 2 reflections = 8 symmetries. Return the
number of augmented (state, pi) pairs.

> **Hint**: 4 rotations × 2 flips = 8.

In [ ]:
state = np.zeros((3, 3), dtype=np.int8)
pi = np.ones(9) / 9.0
n = sol.exercise_3_n_symmetries(state, pi)
print('# symmetries =', n)
assert n == 8


## Exercise 4 — MCTS visit distribution (★★)

Build a fresh `AZNetwork`, run MCTS for 30 simulations from the empty
board, return the visit distribution. It should be a valid probability
vector.

> **Hint**: Inspect legal-action masking and visit normalisation. An untrained network does not guarantee the centre gets the most visits; its random prior and search outcomes determine the result.

In [ ]:
pi = sol.exercise_4_mcts_visits(n_simulations=30, seed=0)
print('visit distribution:', np.round(pi, 3))
assert abs(pi.sum() - 1.0) < 1e-6


## Exercise 5 — AlphaZero winrate (★★)

Train AlphaZero for 5 short iterations; report winrate vs random opponent.
Report the measured score rate, counting draws as 0.5, with its evaluation game count. A short training budget does not guarantee a score above 0.5; compare with an untrained search baseline using the same simulation budget.

> **Hint**: the full pipeline is in `project.train.train_alphazero`.

In [ ]:
wr = sol.exercise_5_alphazero_winrate(n_iters=5, seed=0)
print('AZ winrate:', round(wr, 2))
assert 0 <= wr <= 1


## Exercise 6 — Generalised Advantage Estimation (★★)

Implement GAE on a simple sequence of rewards $r$ and value estimates $V$.
Verify that returns equal advantages + values.

> **Hint**: $A_t = \delta_t + \gamma \lambda A_{t+1}$, $\delta_t = r_t + \gamma V_{t+1} - V_t$.

In [ ]:
adv, ret = sol.exercise_6_gae([1, 0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5, 0.5])
print('advantages:', np.round(adv, 3))
print('returns:', np.round(ret, 3))
assert np.allclose(ret - adv, 0.5)


## Exercise 7 — MAPPO on the coordination game (★★)

Train MAPPO on a 2-agent matrix coordination game (3 actions, 5 steps,
reward = +1 if actions match). Verify mean of last 30 episodes is at least
5 (out of max 10).

> **Hint**: in equilibrium both agents should always pick the same action; perfect coordination yields return = 2 × 5 = 10.

In [ ]:
ret_mean = sol.exercise_7_coord_game_return(n_episodes=200, seed=0)
print('mean return last 30 eps:', round(ret_mean, 2))
assert np.isfinite(ret_mean)


## Exercise 8 — MAPPO on SimpleSpread (★★)

Train MAPPO for a short run on the 3-agent SimpleSpread environment.
Compare first-10 and last-10 episode means. Report the exact interaction and evaluation budgets. This short demonstration cannot establish convergence or a universal episode requirement.

> **Hint**: published SimpleSpread results need ~10k episodes; this is a snapshot of early learning.

In [ ]:
first, last = sol.exercise_8_simple_spread(n_episodes=30, seed=0)
print(f'first 10 mean: {first:.1f}, last 10 mean: {last:.1f}')


## Exercise 9 — Self-play vs. an untrained search baseline (★★★, open-ended)

Compare a self-play-trained network with an untrained network, both using
MCTS. Explain why this is not a controlled comparison against training on
a fixed opponent, and why random-opponent score rate does not establish
an Elo rating or optimal play.

> **Hint**: the moving-target problem; if the opponent never improves, your policy plateaus at ``barely beating the weak opponent''.

In [ ]:
d = sol.exercise_9_self_play_vs_fixed(seed=0)
print('self_play_winrate:', d['self_play_winrate'])
print('untrained_winrate:', d['untrained_winrate'])


**Discussion:** Self-play continuously raises the bar — every iteration the
opponent is at the same level as your current policy. Against a fixed weak
opponent, your policy converges to ``the simplest policy that beats this
opponent'', which can be arbitrarily far from a Nash strategy that
generalises.

## Exercise 10 — AlphaZero random-opponent score trace (★★★, open-ended)

Track score rate against random across iterations, including declines. Compare fixed MCTS simulation budgets; this metric is not an Elo rating. Separate search computation from training interactions.

> **Hint**: more simulations → bigger policy improvement per iteration → steeper climb. But also: more compute per iteration → fewer iterations in fixed time.

In [ ]:
wrs = sol.exercise_10_az_rating_climb(n_iters=6, seed=0)
print('winrates:', np.round(wrs, 2))
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(np.arange(1, len(wrs)+1), wrs, 'o-')
plt.axhline(0.5, color='gray', linestyle='--', label='random')
plt.xlabel('iteration'); plt.ylabel('winrate')
plt.title('AlphaZero rating climb on Tic-Tac-Toe')
plt.legend()
plt.show()


## Wrap-up

Record what you actually demonstrated:
1. Whether you implemented PUCT/MCTS or inspected the reference implementation.
2. The measured random-opponent score rate and search budget.
3. Which inputs the shared actor and centralised critic received.
4. The observed coordination-game return across the runs you executed; a short run is not a convergence proof.
5. Studied the failure modes of MARL: non-stationarity, cycle dynamics,
   coordination failure.

Next week: offline RL (CQL, IQL, BCQ) --- moving from interactive learning
back to learning from logs.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 — PUCT formula (★)</summary>

> **Hint**: Q is negated because the child's value is from the opponent's POV.

</details>

<details>
<summary>Exercise 2 — Terminal value (★)</summary>

> **Hint**: see `project.tictactoe.TicTacToe.terminal_value`.

</details>

<details>
<summary>Exercise 3 — D4 symmetries (★)</summary>

> **Hint**: 4 rotations × 2 flips = 8.

</details>

<details>
<summary>Exercise 4 — MCTS visit distribution (★★)</summary>

> **Hint**: Inspect legal-action masking and visit normalisation. An untrained network does not guarantee the centre gets the most visits; its random prior and search outcomes determine the result.

</details>

<details>
<summary>Exercise 5 — AlphaZero winrate (★★)</summary>

> **Hint**: the full pipeline is in `project.train.train_alphazero`.

</details>

<details>
<summary>Exercise 6 — Generalised Advantage Estimation (★★)</summary>

> **Hint**: $A_t = \delta_t + \gamma \lambda A_{t+1}$, $\delta_t = r_t + \gamma V_{t+1} - V_t$.

</details>

<details>
<summary>Exercise 7 — MAPPO on the coordination game (★★)</summary>

> **Hint**: in equilibrium both agents should always pick the same action; perfect coordination yields return = 2 × 5 = 10.

</details>

<details>
<summary>Exercise 8 — MAPPO on SimpleSpread (★★)</summary>

> **Hint**: published SimpleSpread results need ~10k episodes; this is a snapshot of early learning.

</details>

<details>
<summary>Exercise 9 — Self-play vs. an untrained search baseline (★★★, open-ended)</summary>

> **Hint**: the moving-target problem; if the opponent never improves, your policy plateaus at ``barely beating the weak opponent''.

</details>

<details>
<summary>Exercise 10 — AlphaZero random-opponent score trace (★★★, open-ended)</summary>

> **Hint**: more simulations → bigger policy improvement per iteration → steeper climb. But also: more compute per iteration → fewer iterations in fixed time.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.


## Statistical investigation — A game score depends on the opponent population

**Goal.** Separate player strength from the opponent mix in an aggregate match score. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** A frozen player faces weak and strong opponents. Its win/draw/loss
probabilities are (0.7,0.2,0.1) against weak opponents and (0.2,0.2,0.6)
against strong ones. Predict how its aggregate score changes when the
opponent mix changes, even though the player does not learn.

**Work a small case by hand.** A weak-opponent match has expected score 0.7×1+0.2×0.5+0.1×0=0.8;
a strong-opponent match has 0.2+0.2×0.5=0.3. If w is the weak-opponent
fraction, the unchanged player's expected score is
0.8w+0.3(1-w)=0.3+0.5w. At w=0.2,0.5,0.8 the exact values are
0.4,0.55,0.7. This change requires no learning by the player.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/04-intervals-comparisons.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import independent_mean_interval
rng_stats=np.random.default_rng(113);mixes=[.2,.5,.8];fig,axes=plt.subplots(1,2,figsize=(9,3.3))
for mix in mixes:
    weak=rng_stats.random(300)<mix
    score=np.empty(300)
    for i,is_weak in enumerate(weak):
        p=[.7,.2,.1] if is_weak else [.2,.2,.6]
        score[i]=rng_stats.choice([1.,.5,0.],p=p)
    summary=independent_mean_interval(score,unit='independent match against sampled opponent')
    exact=mix*.8+(1-mix)*.3
    print('Weak-opponent fraction',mix,'exact expected score',exact,'observed summary',summary)
    axes[0].errorbar(mix,summary['estimate'],yerr=[[summary['estimate']-summary['low']],[summary['high']-summary['estimate']]],fmt='o',capsize=4)
    axes[1].scatter(mix,exact,color=RL_PALETTE['red'])
axes[0].set(xlabel='Weak-opponent fraction',ylabel='Mean match score ± 95% t interval')
axes[1].plot(mixes,[m*.8+(1-m)*.3 for m in mixes]);axes[1].set(xlabel='Weak-opponent fraction',ylabel='Exact expected score of unchanged player')
fig.tight_layout();plt.show()


**Interpret and transfer.** Score is a property of a player and evaluation distribution. Self-play
training games do not automatically provide an independent, fixed-opponent
evaluation. Rating comparisons need stated opponents, sides, draws, and sample sizes.

**Transfer.** Evaluate every opening twice with colors reversed. Explain when
the opening pair, rather than individual games, becomes the independent unit.


**Read your result.** Compare each observed interval with the right-panel exact expectation under
the same opponent mix. A rising score across changed mixes cannot by itself
identify an improvement in player strength.

<details><summary>Worked feedback for the changed assumption</summary>

For mirrored openings, summarize the two colors within each opening pair
and resample independent pairs intact. This preserves side balance and the
shared opening effect. State the fixed opponent population; if openings or
opponents share additional dependence, a pair-level IID assumption may still
be insufficient.

</details>
